In [1]:
%pip install matplotlib

Note: you may need to restart the kernel to use updated packages.


In [2]:
# ============================================================
# notebooks/08_fusion_model_training.py  — INDUSTRY VERSION
#
# WHAT CHANGED FROM PREVIOUS VERSION:
#   1. Focal Loss replaces BCEWithLogitsLoss.
#      Forces model to focus on hard fraud cases, not easy legit ones.
#      Directly reduces false positives and improves precision.
#
#   2. Temporal split replaces random split.
#      Train on earliest 70% of time, validate on next 15%,
#      test on most recent 15%. Prevents future data leaking into
#      training — the standard in industry fraud detection.
#
#   3. 13 input features (was 8).
#      FusionNet input_dim auto-detected from the feature matrix.
#
#   4. P1 score regeneration — Stage 1 now forces regeneration
#      if std <= 0.05 regardless of file existence. Deletes the
#      bad 0.011-std file automatically before loading.
#
# EXPECTED RESULTS:
#   AUC:  0.83 – 0.90  (was 0.7537)
#   MCC:  0.35 – 0.55  (was 0.1708)
#   FPR:  < 10%         (was 15.49%)
# ============================================================

import os, json, time, warnings
import numpy as np
import pandas as pd
import joblib
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (roc_auc_score, matthews_corrcoef,
                              precision_score, recall_score, f1_score,
                              roc_curve, confusion_matrix,
                              precision_recall_curve, average_precision_score)

from pathlib import Path
ROOT = str(Path.cwd().parent)
DATA_DIR    = os.path.join(ROOT, "data")
MODEL_DIR   = os.path.join(ROOT, "models")
RESULTS_DIR = os.path.join(ROOT, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)
DEVICE = torch.device("cpu")

TX_PATH   = os.path.join(DATA_DIR, "IEEE CIS", "train_transaction.csv")
SNAP_PATH = os.path.join(DATA_DIR, "tx_snapshot.parquet")
CTX_PATH  = os.path.join(DATA_DIR, "contextual_features.npy")
P1_PATH   = os.path.join(DATA_DIR, "model_probs_full.npy")

print(f"[NB08] Root  : {ROOT}")
print(f"[NB08] Device: {DEVICE}")

# ════════════════════════════════════════════════════════════
# MODEL CLASS DEFINITIONS
# ════════════════════════════════════════════════════════════

SEQ_LEN = 8; CARDINALITY = 4

class FraudAutoencoder(nn.Module):
    def __init__(self, input_dim=224, latent_dim=64):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim,256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(256,128),       nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128,latent_dim))
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim,128), nn.BatchNorm1d(128), nn.ReLU(),
            nn.Linear(128,256),        nn.BatchNorm1d(256), nn.ReLU(),
            nn.Linear(256,input_dim))
    def forward(self, x):
        z = self.encoder(x); return self.decoder(z), z
    def encode(self, x): return self.encoder(x)

class _ResBlock(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.block    = nn.Sequential(
            nn.Linear(in_dim,out_dim), nn.BatchNorm1d(out_dim), nn.ReLU(),
            nn.Linear(out_dim,out_dim), nn.BatchNorm1d(out_dim))
        self.shortcut = nn.Linear(in_dim,out_dim) if in_dim!=out_dim else nn.Identity()
        self.relu     = nn.ReLU()
    def forward(self, x): return self.relu(self.block(x)+self.shortcut(x))

class FraudResNet(nn.Module):
    def __init__(self, input_dim=224):
        super().__init__()
        self.stem   = nn.Linear(input_dim, 128)
        self.blocks = nn.Sequential(
            _ResBlock(128,128), _ResBlock(128,64), _ResBlock(64,64), _ResBlock(64,64))
        self.head = nn.Linear(64, 2)
    def extract(self, x): return self.blocks(torch.relu(self.stem(x)))
    def forward(self, x): return self.head(self.extract(x))

class _ResNeXtBlock(nn.Module):
    def __init__(self, in_dim, out_dim, cardinality=CARDINALITY):
        super().__init__()
        gd = out_dim // cardinality
        self.paths    = nn.ModuleList([
            nn.Sequential(nn.Linear(in_dim,gd), nn.BatchNorm1d(gd), nn.ReLU(),
                          nn.Linear(gd,gd), nn.BatchNorm1d(gd))
            for _ in range(cardinality)])
        self.shortcut = nn.Linear(in_dim,out_dim) if in_dim!=out_dim else nn.Identity()
        self.relu     = nn.ReLU()
    def forward(self, x):
        return self.relu(torch.cat([p(x) for p in self.paths],dim=-1)+self.shortcut(x))

class _ResNeXtExtractor(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            _ResNeXtBlock(input_dim,128), _ResNeXtBlock(128,128),
            _ResNeXtBlock(128,64), _ResNeXtBlock(64,64))
    def forward(self, x): return self.net(x)

class _SelfAttentionGRU(nn.Module):
    def __init__(self, input_dim=64, hidden_dim=64, seq_len=SEQ_LEN):
        super().__init__()
        self.seq_len  = seq_len
        self.step_dim = input_dim // seq_len
        self.gru      = nn.GRU(self.step_dim, hidden_dim, num_layers=2,
                                batch_first=True, dropout=0.3)
        self.attention  = nn.Sequential(nn.Linear(hidden_dim,32), nn.Tanh(), nn.Linear(32,1))
        self.classifier = nn.Sequential(nn.Linear(hidden_dim,32), nn.ReLU(),
                                        nn.Dropout(0.3), nn.Linear(32,1), nn.Identity())
    def forward(self, x):
        x = x.view(x.size(0), self.seq_len, self.step_dim)
        out, _ = self.gru(x)
        w = torch.softmax(self.attention(out), dim=1)
        return torch.sigmoid(self.classifier((w*out).sum(dim=1))).squeeze(1), w

class AttentionRXTJ(nn.Module):
    def __init__(self, input_dim, seq_len=SEQ_LEN):
        super().__init__()
        self.resnext  = _ResNeXtExtractor(input_dim)
        self.attn_gru = _SelfAttentionGRU(input_dim=64, seq_len=seq_len)
    def forward(self, x): return self.attn_gru(self.resnext(x))


# ── FusionNet v3: wider, input_dim auto-detected from feature matrix ─────────
class FusionNet(nn.Module):
    """Attention-weighted MLP. input_dim is set at runtime from feature matrix."""
    def __init__(self, input_dim):
        super().__init__()
        self.feature_attn = nn.Parameter(torch.ones(input_dim))
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 64),        nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 32),         nn.ReLU(),
            nn.Linear(32, 1))
    def forward(self, x):
        w = torch.softmax(self.feature_attn, dim=0)
        return torch.sigmoid(self.net(x * w)).squeeze(1), w

def _logits(m, x):
    return m.net(x * torch.softmax(m.feature_attn, dim=0)).squeeze(1)


# ── Focal Loss: forces model to focus on hard fraud cases ────────────────────
class FocalLoss(nn.Module):
    """
    Focal Loss — Lin et al. 2017 (RetinaNet).
    Reduces the relative loss for well-classified examples so training
    focuses on hard misclassifications (the rare, hard-to-detect fraud cases).

    alpha: weight for minority class (fraud). Use 0.75 for 3.5% fraud rate.
    gamma: focusing strength. 0=standard BCE, 2=standard focal.
    """
    def __init__(self, alpha=0.75, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        bce     = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        probs   = torch.sigmoid(logits)
        p_t     = probs * targets + (1 - probs) * (1 - targets)
        alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        loss    = alpha_t * (1 - p_t) ** self.gamma * bce
        return loss.mean()


# ════════════════════════════════════════════════════════════
# STAGE 1 — Generate real Phase 1 risk scores
# ════════════════════════════════════════════════════════════

# Force regeneration if stale (std <= 0.05 means near-constant = broken)
_need_p1 = True
if os.path.exists(P1_PATH):
    _ex  = np.load(P1_PATH)
    _std = float(np.std(_ex))
    _n   = len(pd.read_parquet(SNAP_PATH, columns=["TransactionID"]))
    if len(_ex) == _n and _std > 0.05:
        print(f"\n[STAGE 1] Valid scores found (n={len(_ex):,}, std={_std:.4f}) — skipping")
        p1_scores = _ex.astype(np.float32)
        _need_p1  = False
    else:
        print(f"\n[STAGE 1] Stale scores (std={_std:.4f} ≤ 0.05) — deleting and regenerating")
        os.remove(P1_PATH)

if _need_p1:
    print("\n" + "="*60)
    print("  STAGE 1: Generating P1 scores — all 224 CSV features")
    print("="*60)

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        imputer  = joblib.load(os.path.join(MODEL_DIR, "imputer.pkl"))
        scaler   = joblib.load(os.path.join(MODEL_DIR, "scaler.pkl"))
        nystroem = joblib.load(os.path.join(MODEL_DIR, "nystroem.pkl"))
        ipca     = joblib.load(os.path.join(MODEL_DIR, "incremental_pca.pkl"))
        ifm      = joblib.load(os.path.join(MODEL_DIR, "isolation_forest.pkl"))

    NYS_DIM  = int(nystroem.n_features_in_)
    IPCA_DIM = int(ipca.n_components_)

    ae_state = torch.load(os.path.join(MODEL_DIR,"autoencoder.pt"), map_location="cpu")
    ae_in    = ae_state["encoder.0.weight"].shape[1]
    ae_model = FraudAutoencoder(input_dim=ae_in)
    ae_model.load_state_dict(ae_state); ae_model.eval()

    rn_model = FraudResNet(input_dim=ae_in)
    rn_model.load_state_dict(
        torch.load(os.path.join(MODEL_DIR,"resnet_extractor.pt"),map_location="cpu"),
        strict=False); rn_model.eval()

    p1_model = AttentionRXTJ(input_dim=IPCA_DIM).to(DEVICE)
    p1_model.load_state_dict(
        torch.load(os.path.join(MODEL_DIR,"attention_rxtj.pt"),map_location=DEVICE))
    p1_model.eval()

    cfg_p1    = json.load(open(os.path.join(RESULTS_DIR,"deployment_config.json")))
    W_MODEL   = float(cfg_p1["W_MODEL"])
    W_IFM     = float(cfg_p1["W_IFM"])
    THRESHOLD = float(cfg_p1["THRESHOLD"])
    print(f"  All models loaded ✓")

    # Load all 224 named columns from original CSV
    print(f"\n[STAGE 1] Loading all 224 features from train_transaction.csv...")
    feature_cols = list(imputer.feature_names_in_)
    load_cols    = ["TransactionID"] + [c for c in feature_cols if c != "TransactionID"]
    t0 = time.time()
    df_full = pd.read_csv(TX_PATH, usecols=load_cols)
    print(f"  Loaded {len(df_full):,} rows ({time.time()-t0:.1f}s)")

    for col in feature_cols:
        if col in df_full.columns and df_full[col].dtype == object:
            df_full[col] = df_full[col].astype("category").cat.codes.astype(np.float32)
            df_full[col] = df_full[col].replace(-1, np.nan)

    snap_ids = pd.read_parquet(SNAP_PATH, columns=["TransactionID"])
    df_full  = snap_ids.merge(df_full, on="TransactionID", how="left")
    X_raw    = df_full[feature_cols].values.astype(np.float32)
    nan_pct  = 100.0 * np.isnan(X_raw).mean()
    print(f"  X_raw: {X_raw.shape}  NaN%: {nan_pct:.1f}%")

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        X_sc = scaler.transform(imputer.transform(X_raw)).astype(np.float32)

    BATCH     = 4096
    Z_ae, Z_rn = [], []
    with torch.no_grad():
        for s in range(0, len(X_sc), BATCH):
            xb = torch.FloatTensor(X_sc[s:s+BATCH])
            Z_ae.append(ae_model.encode(xb).cpu().numpy())
            Z_rn.append(rn_model.extract(xb).cpu().numpy())
            if (s//BATCH) % 30 == 0:
                print(f"  EARN+: {min(s+BATCH,len(X_sc)):>7,}/{len(X_sc):,}")

    Z_earn = np.hstack([np.vstack(Z_ae), np.vstack(Z_rn)]).astype(np.float32)
    assert Z_earn.shape[1] == NYS_DIM, f"EARN dim {Z_earn.shape[1]} ≠ {NYS_DIM}"

    CHUNK  = 8000
    X_ipca = np.zeros((len(Z_earn), IPCA_DIM), dtype=np.float32)
    for s in range(0, len(Z_earn), CHUNK):
        e = min(s+CHUNK, len(Z_earn))
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            X_ipca[s:e] = ipca.transform(nystroem.transform(Z_earn[s:e])).astype(np.float32)

    all_risk = []
    with torch.no_grad():
        for s in range(0, len(X_ipca), BATCH):
            e = min(s+BATCH, len(X_ipca))
            probs, _ = p1_model(torch.FloatTensor(X_ipca[s:e]).to(DEVICE))
            ifm_norm = 1.0 / (1.0 + np.exp(ifm.decision_function(X_ipca[s:e])))
            all_risk.extend((W_MODEL*probs.cpu().numpy() + W_IFM*ifm_norm).tolist())
            if (s//BATCH) % 50 == 0: print(f"  Scoring: {e:>7,}/{len(X_ipca):,}")

    p1_scores = np.array(all_risk, dtype=np.float32)
    print(f"\n  P1: min={p1_scores.min():.4f}  mean={p1_scores.mean():.4f}  "
          f"max={p1_scores.max():.4f}  std={p1_scores.std():.4f}")
    print(f"  Fraud rate @ threshold: {(p1_scores>=THRESHOLD).mean()*100:.2f}%")

    np.save(P1_PATH, p1_scores)

# Patch col-7 with real P1 scores
X_ctx       = np.load(CTX_PATH)
X_ctx[:, 7] = p1_scores
np.save(CTX_PATH, X_ctx)
print(f"\n  contextual_features.npy col-7 patched "
      f"({X_ctx[:,7].min():.4f}–{X_ctx[:,7].max():.4f})")


# ════════════════════════════════════════════════════════════
# STAGE 2 — Train FusionNet v3 with Focal Loss + Temporal Split
# ════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("  STAGE 2: FusionNet v3  |  Focal Loss  |  Temporal Split")
print("="*60)

X        = np.load(CTX_PATH)
y        = np.load(os.path.join(DATA_DIR, "fusion_labels.npy"))
acct_ids = np.load(os.path.join(DATA_DIR, "fusion_account_ids.npy"))
with open(os.path.join(DATA_DIR, "fusion_feature_names.json")) as f:
    FEATURE_NAMES = json.load(f)

print(f"\n  X={X.shape}  positives={int(y.sum()):,} ({100*y.mean():.2f}%)")
for i, name in enumerate(FEATURE_NAMES):
    c = X[:, i]
    print(f"    {name:<25}  range={c.min():.3f}–{c.max():.3f}  "
          f"corr={float(np.corrcoef(c,y)[0,1]):+.4f}")
X = np.nan_to_num(X, nan=0.0)

# ── Temporal split ────────────────────────────────────────────────────────────
# Load TransactionDT from snapshot to split by time, not randomly.
# Industry standard: train on past, evaluate on future.
# Prevents accounts' future transactions from leaking into training.
print("\n[STAGE 2] Building temporal split (70 / 15 / 15 by TransactionDT)...")
snap_dt = pd.read_parquet(SNAP_PATH, columns=["TransactionDT"])["TransactionDT"].values

dt_sorted = np.sort(snap_dt)
t_train   = np.percentile(dt_sorted, 70)
t_val     = np.percentile(dt_sorted, 85)

tr_m = snap_dt <= t_train
va_m = (snap_dt > t_train) & (snap_dt <= t_val)
te_m = snap_dt > t_val

print(f"  Train: {tr_m.sum():,} rows  fraud={y[tr_m].mean()*100:.2f}%")
print(f"  Val  : {va_m.sum():,} rows  fraud={y[va_m].mean()*100:.2f}%")
print(f"  Test : {te_m.sum():,} rows  fraud={y[te_m].mean()*100:.2f}%")
print(f"  (Random split would show ~3.5% fraud in each — "
      f"temporal split shows real distribution over time)")

# Scale AFTER split — fit on train only, transform val/test
from sklearn.preprocessing import StandardScaler as SS
fusion_scaler       = SS()
X_tr_raw, y_tr = X[tr_m], y[tr_m]
X_va_raw, y_va = X[va_m], y[va_m]
X_te_raw, y_te = X[te_m], y[te_m]

X_tr = fusion_scaler.fit_transform(X_tr_raw).astype(np.float32)
X_va = fusion_scaler.transform(X_va_raw).astype(np.float32)
X_te = fusion_scaler.transform(X_te_raw).astype(np.float32)
joblib.dump(fusion_scaler, os.path.join(MODEL_DIR, "fusion_scaler.pkl"))
print("  fusion_scaler fit on train only ✓")

# ── FusionNet setup ───────────────────────────────────────────────────────────
INPUT_DIM = X_tr.shape[1]   # 13 (auto-detected)
model_f   = FusionNet(INPUT_DIM).to(DEVICE)

criterion = FocalLoss(alpha=0.75, gamma=2.0)   # focal loss — better precision
optimizer = torch.optim.Adam(model_f.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=200, eta_min=1e-5)

BATCH_SIZE = 4096
EPOCHS     = 200
PATIENCE   = 25

print(f"\n[STAGE 2] FusionNet v3  input={INPUT_DIM}  "
      f"(13 features, wider 13→128→64→32→1)")
print(f"  Loss: FocalLoss(alpha=0.75, gamma=2.0)")
print(f"  Split: temporal (not random)")

loader = DataLoader(
    TensorDataset(torch.FloatTensor(X_tr), torch.FloatTensor(y_tr)),
    batch_size=BATCH_SIZE, shuffle=True)
Xv_t = torch.FloatTensor(X_va).to(DEVICE)

# ── Training loop ─────────────────────────────────────────────────────────────
print(f"\n[STAGE 2] Training... (target AUC ≥ 0.85)")
best_auc, best_state, no_imp = 0.0, None, 0
train_losses, val_aucs = [], []
t_tr = time.time()

for ep in range(1, EPOCHS + 1):
    model_f.train()
    ep_loss = 0.0
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(_logits(model_f, xb), yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model_f.parameters(), 1.0)
        optimizer.step()
        ep_loss += loss.item() * len(xb)
    scheduler.step()

    model_f.eval()
    with torch.no_grad():
        vp = torch.sigmoid(_logits(model_f, Xv_t)).cpu().numpy()
    vauc = roc_auc_score(y_va, vp)
    train_losses.append(ep_loss / len(X_tr))
    val_aucs.append(vauc)

    if ep % 10 == 0 or ep <= 5:
        print(f"  Epoch {ep:>3}  loss={ep_loss/len(X_tr):.4f}  "
              f"AUC={vauc:.4f}  lr={optimizer.param_groups[0]['lr']:.6f}")

    if vauc > best_auc:
        best_auc = vauc; no_imp = 0
        best_state = {k: v.clone() for k, v in model_f.state_dict().items()}
    else:
        no_imp += 1
        if no_imp >= PATIENCE:
            print(f"  Early stopping at epoch {ep}")
            break

model_f.load_state_dict(best_state)
model_f.eval()
print(f"\n  Best val AUC: {best_auc:.4f}  ({time.time()-t_tr:.0f}s)")

# ── Jaya threshold optimisation ───────────────────────────────────────────────
print("\n[STAGE 2] Jaya threshold optimisation...")
with torch.no_grad():
    vp_np = torch.sigmoid(_logits(model_f, Xv_t)).cpu().numpy()

def _cost(t, p, l):
    pred=(p>=t).astype(int)
    fp=((pred==1)&(l==0)).sum(); fn=((pred==0)&(l==1)).sum()
    tp=((pred==1)&(l==1)).sum(); tn=((pred==0)&(l==0)).sum()
    return 2.0*fp/(fp+tn+1e-9) + fn/(fn+tp+1e-9)

pop = np.random.uniform(0.1, 0.9, 40)
c   = np.array([_cost(t, vp_np, y_va) for t in pop])
for _ in range(150):
    bi,wi = np.argmin(c), np.argmax(c)
    r1,r2 = np.random.rand(40), np.random.rand(40)
    np2   = np.clip(pop+r1*(pop[bi]-np.abs(pop))-r2*(pop[wi]-np.abs(pop)), 0.05, 0.95)
    nc    = np.array([_cost(t, vp_np, y_va) for t in np2])
    m     = nc < c; pop = np.where(m, np2, pop); c = np.where(m, nc, c)

OPT_T  = float(pop[np.argmin(c)])
HIGH_T = min(OPT_T + 0.15, 0.90)
ELEV_T = max(OPT_T - 0.10, 0.25)
print(f"  Optimal={OPT_T:.4f}  HIGH={HIGH_T:.4f}  ELEVATED={ELEV_T:.4f}")

# ── Test evaluation ───────────────────────────────────────────────────────────
print("\n[STAGE 2] Test set evaluation...")
Xt_t = torch.FloatTensor(X_te).to(DEVICE)
with torch.no_grad():
    tp_p = torch.sigmoid(_logits(model_f, Xt_t)).cpu().numpy()
    attn = torch.softmax(model_f.feature_attn, dim=0).cpu().numpy()

preds = (tp_p >= OPT_T).astype(int)
auc   = roc_auc_score(y_te, tp_p)
ap    = average_precision_score(y_te, tp_p)   # precision-recall AUC
mcc   = matthews_corrcoef(y_te, preds)
prec  = precision_score(y_te, preds, zero_division=0)
rec   = recall_score(y_te, preds, zero_division=0)
f1    = f1_score(y_te, preds, zero_division=0)
cm    = confusion_matrix(y_te, preds)
TP,FP = int(cm[1,1]), int(cm[0,1])
FN,TN = int(cm[1,0]), int(cm[0,0])
fpr   = FP / (FP + TN + 1e-9)

print(f"\n  ┌─────────────────────────────────────────────┐")
print(f"  │  AUC-ROC   : {auc:.4f}   (target ≥ 0.85)      │")
print(f"  │  PR-AUC    : {ap:.4f}   (precision-recall)    │")
print(f"  │  MCC       : {mcc:.4f}   (target ≥ 0.35)      │")
print(f"  │  Precision : {prec:.4f}   (target ≥ 0.15)      │")
print(f"  │  Recall    : {rec:.4f}                         │")
print(f"  │  F1        : {f1:.4f}                         │")
print(f"  │  FPR       : {fpr:.4f}   (target ≤ 0.10)      │")
print(f"  │  TP={TP:<5}  FP={FP:<6}  TN={TN:<6}  FN={FN:<5} │")
print(f"  └─────────────────────────────────────────────┘")

print(f"\n  Learned attention weights:")
for i, name in enumerate(FEATURE_NAMES):
    bar = "█" * int(attn[i] * 60)
    print(f"    {name:<25}  {attn[i]:.4f}  {bar}")

# ── Plots ─────────────────────────────────────────────────────────────────────
fpr_a, tpr_a, _  = roc_curve(y_te, tp_p)
pre_a, rec_a, _  = precision_recall_curve(y_te, tp_p)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(fpr_a, tpr_a, color="#0099cc", lw=2, label=f"ROC AUC={auc:.4f}")
axes[0].plot([0,1],[0,1],"k--",lw=0.8)
axes[0].axvline(x=0.10, color="#ff5e57", ls=":", label="10% FPR target")
axes[0].set_xlabel("FPR"); axes[0].set_ylabel("TPR")
axes[0].set_title("ROC Curve"); axes[0].legend()

axes[1].plot(rec_a, pre_a, color="#00e5a0", lw=2, label=f"PR AUC={ap:.4f}")
axes[1].axhline(y=0.15, color="#ffb020", ls=":", label="15% precision target")
axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision")
axes[1].set_title("Precision–Recall Curve"); axes[1].legend()

axes[2].bar(FEATURE_NAMES, attn, color="#00d4ff", edgecolor="#0099cc")
axes[2].set_title("Feature Attention Weights")
axes[2].set_ylabel("Weight"); axes[2].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "fusion_roc.png"), dpi=150, bbox_inches="tight")
plt.close()

fig2, ax = plt.subplots(figsize=(8, 4))
ax.plot(train_losses, label="Train loss", color="#0099cc")
ax.plot(val_aucs,     label="Val AUC",   color="#00e5a0")
ax.set_xlabel("Epoch"); ax.set_title("FusionNet v3 Training"); ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "fusion_training_curves.png"),
            dpi=150, bbox_inches="tight")
plt.close()

# ── Save ──────────────────────────────────────────────────────────────────────
torch.save(model_f.state_dict(), os.path.join(MODEL_DIR, "fusion_net.pt"))

fusion_config = {
    "input_dim":            INPUT_DIM,
    "fusion_threshold":     round(OPT_T, 6),
    "high_threshold":       round(HIGH_T, 6),
    "elevated_threshold":   round(ELEV_T, 6),
    "fusion_auc":           round(float(auc), 6),
    "fusion_pr_auc":        round(float(ap), 6),
    "fusion_mcc":           round(float(mcc), 6),
    "fusion_precision":     round(float(prec), 6),
    "fusion_recall":        round(float(rec), 6),
    "fusion_f1":            round(float(f1), 6),
    "fusion_fpr":           round(float(fpr), 6),
    "true_positives":       TP, "false_positives": FP,
    "true_negatives":       TN, "false_negatives": FN,
    "feature_names":        FEATURE_NAMES,
    "optimal_attn_weights": [round(float(w), 6) for w in attn],
    "model_version":        "fusionnet_v3_focal_temporal",
    "training_epochs":      len(train_losses),
    "best_val_auc":         round(float(best_auc), 6),
    "loss_fn":              "FocalLoss(alpha=0.75, gamma=2.0)",
    "split_method":         "temporal (not random)",
    "label_strategy":       "account_compromise_window_24h",
}
with open(os.path.join(RESULTS_DIR, "fusion_config.json"), "w") as f:
    json.dump(fusion_config, f, indent=2)

print(f"\n[NB08] Saved:")
print(f"  models/fusion_net.pt       model_version=fusionnet_v3_focal_temporal")
print(f"  models/fusion_scaler.pkl   fit on train only (temporal)")
print(f"  results/fusion_config.json")
print(f"  results/fusion_roc.png     (ROC + PR curve + attention weights)")
print(f"\n[NB08] ✓  AUC={auc:.4f}  MCC={mcc:.4f}  FPR={fpr:.4f}  PR-AUC={ap:.4f}")
print(f"[NB08] NEXT STEP → merge app_phase2_endpoints.py into app.py")


[NB08] Root  : f:\rxtj_phase_2
[NB08] Device: cpu

[STAGE 1] Stale scores (std=0.0114 ≤ 0.05) — deleting and regenerating

  STAGE 1: Generating P1 scores — all 224 CSV features
  All models loaded ✓

[STAGE 1] Loading all 224 features from train_transaction.csv...
  Loaded 590,540 rows (8.1s)
  X_raw: (590540, 224)  NaN%: 12.2%
  EARN+:   4,096/590,540
  EARN+: 126,976/590,540
  EARN+: 249,856/590,540
  EARN+: 372,736/590,540
  EARN+: 495,616/590,540
  Scoring:   4,096/590,540
  Scoring: 208,896/590,540
  Scoring: 413,696/590,540

  P1: min=0.2887  mean=0.7121  max=0.7449  std=0.0114
  Fraud rate @ threshold: 99.96%

  contextual_features.npy col-7 patched (0.2887–0.7449)

  STAGE 2: FusionNet v3  |  Focal Loss  |  Temporal Split

  X=(590540, 13)  positives=23,400 (3.96%)
    amount_z_score             range=-5.000–5.000  corr=+0.0248
    merchant_novelty           range=0.000–1.000  corr=+0.0148
    geo_displacement           range=0.000–1.000  corr=-0.0045
    hour_deviation       